# Pemeriksaan Skema UNSW NB15 Sebelum Uji Koreksi

Jalankan notebook ini di Colab pada akun Drive yang mempunyai folder `MyDrive/ercyris_siklus2`. Tujuannya memeriksa kecurigaan bahwa nilai hilang 12,5–16,7% per baris berasal dari nama kolom yang salah atau konversi nonnumerik, bukan mutu bukti sebenarnya. Notebook ini tidak melatih model, tidak mengubah sumber, dan belum membuktikan adanya holdout baru. Pembacaan dua CSV dapat berlangsung beberapa menit. Hanya JSON kecil yang disimpan.


In [ ]:
import csv
import hashlib
import json
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
BASE=Path('/content/drive/MyDrive/ercyris_siklus2')
FIRST=BASE/'UNSW-NB15_1.csv'
SECOND=BASE/'UNSW-NB15_2.csv'
OUT=BASE/'audit_koreksi_siklus2'/'unsw_schema_audit.json'
for path in (FIRST,SECOND):
    if not path.is_file():
        raise FileNotFoundError(f'{path} tidak ditemukan; sesuaikan BASE dengan folder Drive Anda')
OUT.parent.mkdir(parents=True,exist_ok=True)
print('Berkas ditemukan. Keduanya dibaca di Drive; log lain tidak dibaca.')


## Pertama periksa baris awal dan label eksplisit

Loader lama memanggil `pd.read_csv()` tanpa definisi `header`. Bila CSV sebenarnya tanpa header, baris data pertama menjadi nama kolom dan kategori nonnumerik dapat keliru dianggap nilai hilang. Hasil di bawah adalah sinyal audit; keputusan koreksi dibuat setelah melihat ringkasan.


In [ ]:
def first_record(path):
    with path.open('r',encoding='utf-8-sig',errors='replace',newline='') as f:
        return next(csv.reader(f))

def record_properties(path):
    rec=first_record(path)
    lower=[x.strip().lower() for x in rec]
    has_label=any(x in {'label','is_attack'} for x in lower)
    label_method='explicit_header' if has_label else 'last_column_fallback'
    # Kriteria ini hanya mendeteksi ketiadaan nama kolom yang diharapkan;
    # jangan menebak skema atau langsung mengubah data.
    likely_data_row=not has_label and len(rec)>0 and lower[-1] in {'0','1'}
    return {
        'file_name':path.name,
        'bytes':path.stat().st_size,
        'first_record_field_count':len(rec),
        'has_explicit_label_column':has_label,
        'label_selection_by_original_loader':label_method,
        'first_record_looks_like_data':likely_data_row,
        'pandas_default_header_field_count':len(pd.read_csv(path,nrows=0).columns),
    }

file1=record_properties(FIRST)
file2=record_properties(SECOND)
print('Sinyal skema:',json.dumps({'file1':file1,'file2':file2},indent=2))


## Profil nilai hilang menurut kolom yang dipakai loader lama


In [ ]:
def audit_original_loader_missing(path, chunk_size=100_000):
    missing_after=Counter()
    missing_raw=Counter()
    n=0
    labels_invalid=0
    missing_per_row=Counter()
    counts_by_column=Counter()
    label_col_used=None
    for chunk in pd.read_csv(path,chunksize=chunk_size,low_memory=False):
        chunk.columns=[str(col).lower().strip() for col in chunk.columns]
        label_col=next((c for c in ['label','is_attack'] if c in chunk.columns),chunk.columns[-1])
        label_col_used=label_col
        numeric_label=pd.to_numeric(chunk[label_col],errors='coerce')
        labels_invalid += int((~numeric_label.isin([0,1])).sum())
        data=chunk.drop(columns=[label_col,'srcip','dstip','attack_cat'],errors='ignore')
        for col in data:
            missing_raw[col] += int(data[col].isna().sum())
        cat=[col for col in ['proto','state','service'] if col in data]
        if cat:
            data.loc[:,cat]=data[cat].fillna('unknown').astype(str)
        numerical=data.drop(columns=cat).apply(pd.to_numeric,errors='coerce')
        numerical=numerical.replace([np.inf,-np.inf],np.nan)
        miss=numerical.isna().sum(axis=1).astype(int)
        missing_per_row.update(miss.value_counts().to_dict())
        for col in numerical:
            counts_by_column[col]+=len(numerical)
            missing_after[col]+=int(numerical[col].isna().sum())
        for col in cat:
            counts_by_column[col]+=len(data)
            missing_after[col]+=0
        n+=len(chunk)
    worst=sorted(missing_after,key=lambda k:(-missing_after[k],str(k)))[:12]
    return {
        'pandas_data_rows':n,
        'effective_features':len(counts_by_column),
        'label_column_selected':label_col_used,
        'invalid_or_missing_binary_labels':labels_invalid,
        'missing_features_per_row_histogram':{str(k):int(v) for k,v in sorted(missing_per_row.items())},
        'top_12_missing_or_non_numeric_columns':[
            {'column':col,'raw_missing':missing_raw[col],
             'after_original_numeric_coercion':missing_after[col],
             'non_numeric_share':missing_after[col]/max(counts_by_column[col],1)}
            for col in worst
        ],
        'columns_missing_in_every_row':sum(missing_after[k]==counts_by_column[k] for k in counts_by_column),
    }

quality=audit_original_loader_missing(FIRST)
print('Profil konversi loader lama:',json.dumps(quality,indent=2,ensure_ascii=False))


## Periksa baris yang identik di berkas 1 dan 2

Hash baris membantu menolak duplikasi tepat, tetapi nilai nol tidak membuktikan independensi waktu atau host. Ringkasan saja yang diekspor.


In [ ]:
def row_digests(path,skip_header):
    with path.open('r',encoding='utf-8-sig',errors='replace',newline='') as f:
        reader=csv.reader(f)
        if skip_header:next(reader,None)
        for row in reader:
            canonical='\x1f'.join(row).encode('utf-8','surrogatepass')
            yield hashlib.blake2b(canonical,digest_size=16).digest()

# Perbandingan byte-kanonik semua kolom dan label; tidak menguji kesamaan
# kejadian, host, atau kedekatan waktu yang tidak byte-identik.
digests1=set(row_digests(FIRST,file1['has_explicit_label_column']))
rows1=sum(1 for _ in row_digests(FIRST,file1['has_explicit_label_column']))
rows2=0
overlap_count=0
for digest in row_digests(SECOND,file2['has_explicit_label_column']):
    rows2+=1
    overlap_count+=digest in digests1

overlap_report={'file1_records':rows1,'file1_unique_record_hashes':len(digests1),
                'file2_records':rows2,'file2_exact_record_hash_overlap':overlap_count,
                'hash': 'BLAKE2b-128 on CSV-parsed row values; no raw rows exported',
                'limitation':'No host/time/group independence is inferred from this check.'}
result={'status':'AUDIT_ONLY_NOT_AN_EXPERIMENT','file1_schema':file1,'file2_schema':file2,
        'original_loader_missingness':quality,'file1_file2_overlap':overlap_report,
        'interpretation_rule':(
            'If first_record_looks_like_data is true, the old loader likely treats '
            'one data row as a header; stop before using UNSW results for new claims. '
            'Missing values from non-numeric coercion are not automatic evidence quality.'
        )}
OUT.write_text(json.dumps(result,indent=2,ensure_ascii=False),encoding='utf-8')
print('HASIL:',OUT)
print('Kirim hanya unsw_schema_audit.json; jangan kirim CSV mentah.')
